[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/josephson/josephson.ipynb)

# Fluxons in a Josephson Junction

This example models magnetic fluxons travelling through a planar Josephson junction as relativistic strings of quantized flux. Across a thin insulating barrier between two superconductors, a supercurrent tunnels in proportion to the sine of the condensate phase difference while a barrier voltage turns that phase difference in time, following Josephson's two relations. Coupling those relations to the magnetic field inside a wide barrier yields the two-dimensional sine-Gordon wave equation, whose kinks carry one flux quantum per unit length and propagate at most at the junction's Swihart light speed. Straight kinks contract as they accelerate, closed loops shrink under their own line tension, and an oblique kink meeting an open boundary reflects as an antifluxon.

Joining the junction's spacetime with an orthogonal phase plane models the local phase difference as a unit direction in that plane and its angular rate as a phase-plane bivector. Tunnelling pulls each phase direction toward the zero-phase axis while triangle simplex gradients couple neighbouring directions across the supercurrent duality map, so a single wedge `phases ^ pull` gives both the sinusoidal Josephson torque and the spatial curvature torque without unwrapping angles. Wedging the triangle simplex gradients with the vertex phase directions forms the mixed spacetime-phase field bivector `gradients ^ corner_phases`, whose squared norm measures the magnetic field energy density. Because the internal phase plane commutes with the junction's spacetime, spacetime boosts and spatial rotations act on junction events by `boost >> events` and `tilt_rotor >> events` while leaving phase directions untouched, following the Stockholm lecture notes on the [Josephson effect](https://ekmf.fysik.su.se/Supraledning/WWW/Lectures/Lecture9.pdf) and on [fluxons in Josephson junctions](https://ekmf.fysik.su.se/Supraledning/WWW/Lectures/Lecture12.pdf).

In scalar sine-Gordon PDE notation, with lengths in Josephson penetration depths and times in inverse plasma frequencies, the unwrapped phase angle $\varphi(x, y, t)$ satisfies $\partial_x^2\varphi + \partial_y^2\varphi - \partial_t^2\varphi = \sin\varphi$ subject to open-edge Neumann conditions $\mathbf{n}\cdot\nabla\varphi = 0$, and an oblique boosted kink requires substituting the rotated and Lorentz-contracted coordinate $\gamma\bigl((x - x_0)\cos\theta + y\sin\theta - u t\bigr)$ with $\gamma = (1 - u^2)^{-1/2}$ into $4\arctan\exp(\cdot)$ while tracking $2\pi$ branch jumps across collisions.

In [ ]:
# Install numga and its examples when running on Google Colab.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from examples.electromagnetism.josephson import render, scenarios
from examples.util.animation import show_states
from numga import Algebra, NumpyContext
from numga.sparse import SparseExtensor, spdiag
from numga.topology import Topology

# Spacetime of the planar junction joined with an orthogonal Euclidean plane for the condensate phase.
phase_plane = Algebra("a+b+")
ga = Algebra("x+y+t-") * phase_plane
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Plane = ga.gatype.vector()                                                    # Vector+
Point = ga.gatype.antivector()                                                # Antivector-
Phase = Plane.derive.within(phase_plane)                                      # Vector+
Rate = ga.gatype.bivector().derive.within(phase_plane)                        # Bivector+
# Phase-plane rotor turning the zero-phase axis into a local phase direction.
Turn = (mv.ab * 0).exp().gatype                                               # Rotor+
# Unit spacetime displacements along the junction plane and along time.
x_step = mv.x.dual()                                                          # [] Point-
y_step = mv.y.dual()                                                          # [] Point-
time_step = -mv.t.dual()                                                      # [] Point-
# Duality map reading a spatial phase gradient as the supercurrent it drives along the junction.
supercurrent = Plane.dual()                                                   # [] Point- <- Plane+

## Pendulum chain collision

Over a triangulated junction, pairing each triangle's corner simplex gradients across `supercurrent` and pulling the coupling back to vertices gives the spatial tension operator on phase directions. Along a narrow strip, each local phase direction acts like a pendulum hanging from a horizontal torsion rod, where Josephson tunnelling supplies gravity and magnetic tension twists neighbouring rods toward alignment. Composing the phase-plane rotors of an approaching fluxon and antifluxon launches opposite full turns toward the centre of the strip. Stepping the narrow strip takes about five seconds.

In [ ]:
def simplex_gradients(topology: Topology, vertices: Point) -> tuple[Scalar, Plane]:
    """Boundary edges and oriented triangle patches give the triangle areas (`Scalar+[F]`) and corner simplex gradients (`[3] Plane+[F]`)."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)              # [3] Point-[F]
    patches = (sides[0] & sides[1]) / 2                                       # Antibivector-[F]
    areas = patches.antinorm()                                                # Scalar+[F]
    gradients = (sides | patches.inverse()) / 2                               # [3] Plane+[F]
    return areas, gradients


def tension_operator(topology: Topology, areas: Scalar, gradients: Plane) -> SparseExtensor:
    """Pulling the triangle gradient-supercurrent pairing back to vertices and dividing by lumped dual areas forms the field tension (`Scalar+[V <- V]`)."""
    corners = topology.incidence[2, 0]                                        # [3] Scalar+[F <- V]
    dual_areas = (corners.adjugate() * (areas / 3)).sum(axis=0)               # Scalar+[V]
    # Pair each corner's simplex gradient with the supercurrent driven by another corner's gradient over the triangle.
    local = (gradients[:, None] & supercurrent(gradients[None, :])) * areas   # [3, 3] Scalar+[F]
    stiffness = (corners[:, None].adjugate() * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)
    return spdiag(1 / dual_areas) * stiffness


strip, strip_coordinates = scenarios.junction_grid(
    scenarios.n_strip_columns, scenarios.n_strip_rows, scenarios.strip_length, scenarios.strip_height
)
strip_vertices = (x_step * strip_coordinates[:, 0] + y_step * strip_coordinates[:, 1]).field()  # Point-[V]
strip_areas, strip_gradients = simplex_gradients(strip, strip_vertices)
strip_tension = tension_operator(strip, strip_areas, strip_gradients)

In [ ]:
def fluxon_turn(events: Point, rapidity: np.ndarray, sense: float) -> Turn:
    """Boosting junction events (`Point-[V]`) into the kink rest frame produces the phase-plane rotors (`[...] Turn+[V]`) of a straight fluxon or antifluxon."""
    # Boost spacetime events into the moving kink's rest frame and read their coordinate normal to the kink line.
    boost = (mv.xt * (-rapidity / 2)).exp()                                   # [...] Rotor+
    rest_offsets = mv.x & (boost >> events)                                   # [...] Scalar+[V]
    angles = 4 * (sense * rest_offsets).exp().arctan()                        # [...] Scalar+[V]
    return (mv.ab * (-angles / 2)).exp()


def phase_frames(tension: SparseExtensor, phases: Phase, rates: Rate, dt: float, n_steps: int,
                 stride: int) -> Iterator[Phase]:
    """Yields the vertex phase directions (`[...] Phase+[V]`) every `stride` leapfrog steps of `dt` under Josephson tunnelling and field tension."""
    for step_index in range(n_steps + 1):
        if step_index % stride == 0:
            yield phases
        pull = mv.a - tension * phases                                        # [...] Phase+[V]
        rates = rates + dt * (phases ^ pull)                                  # [...] Rate+[V]
        step_turns = (rates * (-dt / 2)).exp()                                # [...] Turn+[V]
        phases = step_turns >> phases                                         # [...] Phase+[V]


dt = scenarios.strip_dt
middle_row = np.isclose(strip_coordinates[:, 1], 0.0, atol=1e-9) & (np.abs(strip_coordinates[:, 0]) <= scenarios.pendulum_reach)
pendulum_indices = np.flatnonzero(middle_row)[::scenarios.pendulum_step]
n_pendula = len(pendulum_indices)
strip_pivots = strip_vertices.batch()[pendulum_indices]                       # [n_pendula] Point-
# Multiply the fluxon rotor on the left and the antifluxon rotor on the right so their phase turns add.
first_collision_events = strip_vertices + time_step * 0.0                     # Point-[V]
next_collision_events = strip_vertices + time_step * dt                       # Point-[V]
first_pair_turns = (fluxon_turn(first_collision_events + x_step * scenarios.collision_offset, scenarios.collision_rapidity, 1.0) *
                    fluxon_turn(first_collision_events - x_step * scenarios.collision_offset, -scenarios.collision_rapidity, -1.0))
next_pair_turns = (fluxon_turn(next_collision_events + x_step * scenarios.collision_offset, scenarios.collision_rapidity, 1.0) *
                   fluxon_turn(next_collision_events - x_step * scenarios.collision_offset, -scenarios.collision_rapidity, -1.0))
collision_phases = first_pair_turns >> mv.a                                   # Phase+[V]
collision_rates = (collision_phases ^ (next_pair_turns >> mv.a)) / dt         # Rate+[V]
collision_run = phase_frames(strip_tension, collision_phases, collision_rates, dt, scenarios.n_collision_steps, scenarios.collision_stride)
pendulum_phases = show_states(
    (frame_phases.batch()[pendulum_indices] for frame_phases in collision_run),
    render.pendulum_frames, strip_pivots, name="josephson_pendula", duration_ms=scenarios.frame_duration_ms
);

Viewed obliquely along the strip axis, resting pendula hang straight down while each passing soliton swings them once over the top, coloured red and blue by the signed tunnelling current. When the fluxon from the left meets the antifluxon from the right, their opposite twists pass cleanly through each other and emerge intact.

## Lorentz contraction of moving fluxons

Evolving a batch of straight fluxons at increasing rapidities on the same strip tests relativistic contraction directly on the discrete triangulation. Because every fluxon carries a single quantum of magnetic flux, its spatial width inversely tracks the peak norm of the mixed spacetime-phase gradient bivector at its core. Stepping the boosted batch takes about six seconds.

In [ ]:
def field_strengths(topology: Topology, gradients: Plane, phases: Phase) -> Scalar:
    """Wedging triangle simplex gradients with vertex phase directions gives the squared magnetic field strength (`[...] Scalar+[F]`) in each triangle."""
    corners = topology.incidence[2, 0]                                        # [3] Scalar+[F <- V]
    corner_phases = corners * phases[..., None]                               # [..., 3] Phase+[F]
    fields = (gradients ^ corner_phases).sum(axis=-1)                         # [...] Bivector+[F]
    return fields.scalar_norm_squared()


rapidities = scenarios.rapidities
n_rapidities = scenarios.n_rapidities
shifted_strip = strip_vertices - x_step * scenarios.strip_origin              # Point-[V]
first_boost_turns = fluxon_turn(shifted_strip + time_step * 0.0, rapidities, 1.0)
next_boost_turns = fluxon_turn(shifted_strip + time_step * dt, rapidities, 1.0)
boost_phases = first_boost_turns >> mv.a                                      # [n_rapidities] Phase+[V]
boost_rates = (boost_phases ^ (next_boost_turns >> mv.a)) / dt                # [n_rapidities] Rate+[V]
boost_snapshots = list(phase_frames(strip_tension, boost_phases, boost_rates, dt, scenarios.n_strip_steps, scenarios.n_strip_steps))
initial_strengths = field_strengths(strip, strip_gradients, boost_snapshots[0])  # [n_rapidities] Scalar+[F]
final_strengths = field_strengths(strip, strip_gradients, boost_snapshots[-1])   # [n_rapidities] Scalar+[F]
speeds = np.tanh(rapidities)
render.contraction_figure(speeds, initial_strengths, final_strengths);

Blue circles mark the fluxon widths relative to the resting kink at the start of the run and blue squares mark their widths after propagating along the strip, both following the orange Lorentz contraction curve toward the Swihart velocity.

## Collapsing fluxon ring

Bending a resting fluxon line into a closed circular loop leaves no external pressure to balance its line tension. Released from rest on a square junction, the loop accelerates inward toward the Swihart speed, passes through the origin into an expanding antifluxon ring, and radiates small-amplitude plasma waves.

In [ ]:
square, square_coordinates = scenarios.junction_grid(
    scenarios.n_square_side, scenarios.n_square_side, scenarios.square_side, scenarios.square_side
)
square_vertices = (x_step * square_coordinates[:, 0] + y_step * square_coordinates[:, 1]).field()  # Point-[V]
square_areas, square_gradients = simplex_gradients(square, square_vertices)
square_tension = tension_operator(square, square_areas, square_gradients)
ring_distances = square_vertices.antinorm()                                   # Scalar+[V]
ring_angles = 4 * (ring_distances - scenarios.ring_radius).exp().arctan()     # Scalar+[V]
ring_turns = (mv.ab * (-ring_angles / 2)).exp()                               # Turn+[V]
ring_phases = ring_turns >> mv.a                                              # Phase+[V]
ring_run = phase_frames(square_tension, ring_phases, 0 * (mv.a ^ ring_phases), scenarios.ring_dt, scenarios.n_ring_steps, scenarios.ring_stride)
ring_currents = show_states(
    (mv.a ^ frame_phases for frame_phases in ring_run),
    render.current_frames, square, square_vertices, name="josephson_ring", duration_ms=scenarios.frame_duration_ms
);

Red and blue bands show opposite directions of tunnelling supercurrent on the inner and outer sides of the fluxon loop. As the ring contracts through the centre, its inner and outer current bands invert into an antifluxon ring while circular plasma ripples spread across the junction.

## Antifluxon reflection at an open edge

Rotating junction events in the spatial plane before boosting launches an oblique fluxon line toward the right boundary. Because no supercurrent escapes an open edge, the top and bottom ends bend perpendicular to the horizontal borders while the right border reflects the arriving fluxon as an antifluxon.

In [ ]:
wide, wide_coordinates = scenarios.junction_grid(
    scenarios.n_wide_columns, scenarios.n_wide_rows, scenarios.junction_width, scenarios.junction_height
)
wide_vertices = (x_step * wide_coordinates[:, 0] + y_step * wide_coordinates[:, 1]).field()  # Point-[V]
wide_areas, wide_gradients = simplex_gradients(wide, wide_vertices)
wide_tension = tension_operator(wide, wide_areas, wide_gradients)
tilt_rotor = (mv.xy * (scenarios.line_tilt / 2)).exp()                        # [] Rotor+
shifted_wide = wide_vertices - x_step * scenarios.line_offset                 # Point-[V]
first_line_events = tilt_rotor >> (shifted_wide + time_step * 0.0)            # Point-[V]
next_line_events = tilt_rotor >> (shifted_wide + time_step * scenarios.line_dt)  # Point-[V]
first_line_turns = fluxon_turn(first_line_events, scenarios.line_rapidity, 1.0)  # Turn+[V]
next_line_turns = fluxon_turn(next_line_events, scenarios.line_rapidity, 1.0)    # Turn+[V]
line_phases = first_line_turns >> mv.a                                        # Phase+[V]
line_rates = (line_phases ^ (next_line_turns >> mv.a)) / scenarios.line_dt    # Rate+[V]
line_run = phase_frames(wide_tension, line_phases, line_rates, scenarios.line_dt, scenarios.n_line_steps, scenarios.line_stride)
line_currents = show_states(
    (mv.a ^ frame_phases for frame_phases in line_run),
    render.current_frames, wide, wide_vertices, name="josephson_reflection", duration_ms=scenarios.frame_duration_ms
);

Before reaching the right boundary, the tilted fluxon carries red tunnelling current on its trailing side and blue on its leading side. After bouncing from the open edge, the returning wave travels leftward with blue on its left and red on its right, reversing magnetic polarity into an antifluxon.

In [ ]:
# checks
# Rotor stepping preserves unit phase directions and carries full-turn solitons through the collision.
assert ((pendulum_phases[-1].norm() - 1).abs() < 1e-12).all()
assert (mv.a | pendulum_phases[-1]).min() < -0.9
# Boosted fluxon widths match Lorentz contraction within three percent before and after propagation.
lorentz_factors = 1 / np.cosh(rapidities)
for strengths in (initial_strengths, final_strengths):
    peaks = strengths.batch().max(axis=-1).square_root()
    assert ((peaks[0] / peaks - lorentz_factors).abs() < 0.03 * lorentz_factors).all()
# Line tension shrinks the outer radius of strong tunnelling current before the ring reaches the origin.
radial_distances = ring_distances.batch()
ring_radii = [radial_distances[ring_currents[index].batch().norm() > 0.5].max() for index in (0, 30)]
assert ring_radii[1] < ring_radii[0] - 2
# Open-edge reflection reverses the left-to-right sign order of the tunnelling current along the midline.
middle_mask = (mv.y & wide_vertices.batch()).abs() < 1e-9
middle_x = mv.x & wide_vertices.batch()[middle_mask]
current_dual = mv.ab.dual()
start_current = line_currents[0].batch()[middle_mask] & current_dual
end_current = line_currents[-1].batch()[middle_mask] & current_dual
start_span = middle_x[start_current.argmax()] - middle_x[start_current.argmin()]
end_span = middle_x[end_current.argmax()] - middle_x[end_current.argmin()]
assert start_span * end_span < 0